# Notebook for measuring runtime of Hashing, bucketing and similarity value computation 

In [5]:
import os
import sys
import numpy as np
import itertools
import pandas as pd

def find_project_root(target_folder="masteroppgave"):
    """Find the absolute path of a folder by searching upward."""
    currentdir = os.path.abspath("__file__")  # Get absolute script path
    while True:
        if os.path.basename(currentdir) == target_folder:
            return currentdir  # Found the target folder
        parentdir = os.path.dirname(currentdir)
        if parentdir == currentdir:  # Stop at filesystem root
            return None
        currentdir = parentdir  # Move one level up

# Example usage
project_root = find_project_root("masteroppgave")

if project_root:
    sys.path.append(project_root)
    print(f"Project root found: {project_root}")
else:
    raise RuntimeError("Could not find 'masteroppgave' directory")

from utils.helpers.measure_similarities import *


Project root found: c:\Users\eivin\dev\masteroppgave


# Code for running several combinations of the runtime parameters

# Disk

In [ ]:
MEASURE="disk_dtw_cy"
CITY="porto"
DATA_SIZE = [500]

#Strategies
BUCKETING_METHOD = "loose"
TRUE_TRAJECTORIES = True

#Logistics
PARALLEL_JOBS = 24
ITERATIONS = 1

In [ ]:
# Parameter groups (bulk sets)
bulk_groups = [
    {
        "name": "DIA_0.1-0.4",
        "DIAMETER_VALUES": [0.1, 0.2, 0.3, 0.4],
        "LAYERS_VALUES": [1, 2, 3],
        "DISKS_VALUES": [50, 100, 200, 300, 400, 500]
    },
    {
        "name": "DIA_0.6-0.8",
        "DIAMETER_VALUES": [0.6, 0.8],
        "LAYERS_VALUES": [1, 2, 3],
        "DISKS_VALUES": [50, 100, 150, 200, 250]
    },
    {
        "name": "DIA_1-1.4",
        "DIAMETER_VALUES": [1, 1.2, 1.4],
        "LAYERS_VALUES": [1, 2, 3],
        "DISKS_VALUES": [20, 40, 60, 80, 100]
    },
    {
        "name": "DIA_1.6-1.8",
        "DIAMETER_VALUES": [1.6, 1.8],
        "LAYERS_VALUES": [1, 2, 3],
        "DISKS_VALUES": [10, 20, 30]
    },
    {
        "name": "EDGE_2",
        "DIAMETER_VALUES": [2],
        "LAYERS_VALUES": [1, 2, 3],
        "DISKS_VALUES": [2, 4, 6, 8, 10, 12]
    },
    {
        "name": "EDGE_3",
        "DIAMETER_VALUES": [3],
        "LAYERS_VALUES": [1, 2, 3],
        "DISKS_VALUES": [1, 2, 3, 4, 5]
    },
    {
        "name": "EDGE_4",
        "DIAMETER_VALUES": [4],
        "LAYERS_VALUES": [1, 2, 3],
        "DISKS_VALUES": [1, 2, 3, 4, 5]
    }
]


In [ ]:
if "disk_dtw_cy" or "disk_frechet_cy" in MEASURE:
    SCHEME = "disk"
elif "grid_dtw_cy" or "grid_frechet_cy" in MEASURE:
    SCHEME = "grid"
    
if "dtw" in MEASURE:
    measure = "dtw"
elif "frechet" in MEASURE:
    measure = "frechet"

#Filenames
if TRUE_TRAJECTORIES:
    folder = "true_trajectories"
    file_name = f"runtimes_bucketing({BUCKETING_METHOD})_(true_trajectories)_{CITY}_{measure}_{DATA_SIZE}_{SCHEME}.csv"
else:
    folder = "hashed_trajectories"
    file_name = f"runtimes_bucketing({BUCKETING_METHOD})_(hashed_trajectories)_{CITY}_{measure}_{DATA_SIZE}_{SCHEME}.csv"

output_path = f"../../../results_hashed/runtimes/bucketing/{CITY}/{folder}/{BUCKETING_METHOD}/{measure}/{file_name}"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

In [ ]:
import itertools

print(f" Current param config: \n \tBUCKETING: YES \n\tBUCKETING_METHOD: {BUCKETING_METHOD}\n\tTRUE_TRAJECTORIES: {TRUE_TRAJECTORIES}\n\tCITY: {CITY}\n\tMEASURE: {MEASURE}\n\tDATA_SIZE: {DATA_SIZE}\n\tSCHEME: {SCHEME} \n\tPARALLEL_JOBS: {PARALLEL_JOBS} \n\tITERATIONS: {ITERATIONS} \n\n")

first_write = True  # Write header only once for the whole output file

for group in bulk_groups:
    DIAMETER_LIST = group["DIAMETER_VALUES"]
    LAYERS_LIST = group["LAYERS_VALUES"]
    DISKS_LIST = group["DISKS_VALUES"]

    param_combinations = list(itertools.product(DIAMETER_LIST, LAYERS_LIST, DISKS_LIST, DATA_SIZE))

    for diameter, layers, disks, data_size in param_combinations:
        print(f" \n Running for Diameter: {diameter}, Layers: {layers}, Disks: {disks}, Data Size: {data_size}, True Trajectories: {TRUE_TRAJECTORIES}")

        # Call the appropriate function
        if TRUE_TRAJECTORIES:
            df_result = compute_hashed_similarity_runtimes_with_bucketing_with_true_sim(
                measure=MEASURE,
                city=CITY,
                diameter=diameter,
                layers=layers,
                disks=disks,
                parallel_jobs=PARALLEL_JOBS,
                data_size=data_size,
                iterations=ITERATIONS,
                bucketing_method=BUCKETING_METHOD
            )
        else:
            df_result = compute_hashed_similarity_runtimes_with_bucketing(
                measure=MEASURE,
                city=CITY,
                diameter=diameter,
                layers=layers,
                disks=disks,
                parallel_jobs=PARALLEL_JOBS,
                data_size=data_size,
                iterations=ITERATIONS,
                bucketing_method=BUCKETING_METHOD
            )

        # Add parameters to result DataFrame
        df_result["City"] = CITY
        df_result["Measure"] = measure
        df_result["Diameter"] = diameter
        df_result["Layers"] = layers
        df_result["Disks"] = disks
        df_result["Size"] = data_size

        # Define the desired column order
        desired_order = ["City", "Measure", "Diameter", "Layers", "Disks", "Size",
                         "Average Similarity Computation Time (Seconds)", 
                         "Average Hash Generation Time (Seconds)", 
                         "Average Bucket Distribution Time (Seconds)", 
                         "Total time (Seconds)"]

        df_result = df_result[desired_order]

        # Append to single output file
        df_result.to_csv(output_path, mode='a', header=first_write, index=False)
        first_write = False


# Grid

In [ ]:
MEASURE = "grid_dtw_cy"
CITY = "porto"
DATA_SIZE = [500]

# Strategies
BUCKETING_METHOD = "loose"
TRUE_TRAJECTORIES = True

# Logistics
PARALLEL_JOBS = 24
ITERATIONS = 1

In [7]:
bulk_groups = [
    {
        "name": "RES_0.01-0.15",
        "RESOLUTION_VALUES": [0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1, 0.11, 0.12, 0.13, 0.14, 0.15],
        "LAYERS_VALUES": [1, 2, 3, 4]
    },
    {
        "name": "EDGE",
        "RESOLUTION_VALUES": [0.5, 1],
        "LAYERS_VALUES": [1, 2, 3, 4, 5, 6, 10]
    }
]


In [ ]:
# Determine SCHEME and measure type
if "disk_dtw_cy" in MEASURE or "disk_frechet_cy" in MEASURE:
    SCHEME = "disk"
elif "grid_dtw_cy" in MEASURE or "grid_frechet_cy" in MEASURE:
    SCHEME = "grid"

if "dtw" in MEASURE:
    measure = "dtw"
elif "frechet" in MEASURE:
    measure = "frechet"

# File path setup
if TRUE_TRAJECTORIES:
    folder = "true_trajectories"
    file_name = f"runtimes_bucketing({BUCKETING_METHOD})_(true_trajectories)_{CITY}_{measure}_{DATA_SIZE}_{SCHEME}.csv"
else:
    folder = "hashed_trajectories"
    file_name = f"runtimes_bucketing({BUCKETING_METHOD})_(hashed_trajectories)_{CITY}_{measure}_{DATA_SIZE}_{SCHEME}.csv"

output_path = f"../../../results_hashed/runtimes/bucketing/{CITY}/{folder}/{BUCKETING_METHOD}/{measure}/{file_name}"
os.makedirs(os.path.dirname(output_path), exist_ok=True)


In [ ]:
import itertools

print(f" Current param config: \n \tBUCKETING: YES \n\tBUCKETING_METHOD: {BUCKETING_METHOD}\n\tTRUE_TRAJECTORIES: {TRUE_TRAJECTORIES}\n\tCITY: {CITY}\n\tMEASURE: {MEASURE}\n\tDATA_SIZE: {DATA_SIZE}\n\tSCHEME: {SCHEME} \n\tPARALLEL_JOBS: {PARALLEL_JOBS} \n\tITERATIONS: {ITERATIONS} \n\n")

first_write = True

for group in bulk_groups:
    RESOLUTION_LIST = group["RESOLUTION_VALUES"]
    LAYERS_LIST = group["LAYERS_VALUES"]

    param_combinations = list(itertools.product(RESOLUTION_LIST, LAYERS_LIST, DATA_SIZE))

    for resolution, layers, data_size in param_combinations:
        print(f"\nRunning for Resolution: {resolution}, Layers: {layers}, Data Size: {data_size}, True Trajectories: {TRUE_TRAJECTORIES}")

        if TRUE_TRAJECTORIES:
            df_result = compute_hashed_similarity_runtimes_with_bucketing_with_true_sim(
                measure=MEASURE,
                city=CITY,
                res=resolution,
                layers=layers,
                parallel_jobs=PARALLEL_JOBS,
                data_size=data_size,
                iterations=ITERATIONS,
                bucketing_method=BUCKETING_METHOD
            )
        else:
            df_result = compute_hashed_similarity_runtimes_with_bucketing(
                measure=MEASURE,
                city=CITY,
                res=resolution,
                layers=layers,
                parallel_jobs=PARALLEL_JOBS,
                data_size=data_size,
                iterations=ITERATIONS,
                bucketing_method=BUCKETING_METHOD
            )

        # Add parameters to result DataFrame
        df_result["City"] = CITY
        df_result["Measure"] = measure
        df_result["Resolution"] = resolution
        df_result["Layers"] = layers
        df_result["Size"] = data_size

        # Reorder columns
        desired_order = [
            "City", "Measure", "Resolution", "Layers", "Size",
            "Average Similarity Computation Time (Seconds)",
            "Average Hash Generation Time (Seconds)",
            "Average Bucket Distribution Time (Seconds)",
            "Total time (Seconds)"
        ]
        df_result = df_result[desired_order]

        # Append to file
        df_result.to_csv(output_path, mode='a', header=first_write, index=False)
        first_write = False


 Current param config: 
 	BUCKETING: YES 
	BUCKETING_METHOD: loose
	TRUE_TRAJECTORIES: False
	CITY: porto
	MEASURE: grid_dtw_cy
	DATA_SIZE: [500]
	SCHEME: grid 
	PARALLEL_JOBS: 4 
	ITERATIONS: 1 



Running for Resolution: 0.01, Layers: 1, Data Size: 500, True Trajectories: False
Iteration 1/1

Running for Resolution: 0.01, Layers: 2, Data Size: 500, True Trajectories: False
Iteration 1/1

Running for Resolution: 0.01, Layers: 3, Data Size: 500, True Trajectories: False
Iteration 1/1
